# Análisis exploratorio (EDA) — BTC/USDT (sabana_trading_rl)

Objetivo: entender las características de los datos que afectan al agente y **justificar** la elección de
features, la partición temporal y el diseño de la recompensa.

> Ejecutar desde la raíz del repo o desde `notebooks/`. Requiere `data/BTCUSDT_1d.csv` (`python -m src.data`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_prices, compute_features, build_splits, count_missing_bars
from src.metrics import drawdown_series
from src.utils import load_config

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
cfg = load_config(ROOT / "configs" / "base.yaml")
prices = load_prices(ROOT / cfg["data"]["csv"])
PPY = cfg["env"]["periods_per_year"]   # 365: cripto opera 24/7
SYM = cfg["data"]["symbol"]
splits = build_splits(prices, cfg["data"])
bounds = {k: tuple(pd.Timestamp(x) for x in v) for k, v in cfg["data"]["splits"].items()}
colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:red"}
print(SYM, prices.shape, prices.index.min().date(), "->", prices.index.max().date())
print("Velas faltantes (huecos del exchange):", count_missing_bars(prices, cfg["data"]["download"]["timeframe"]))
print("Velas duplicadas:", int(prices.index.duplicated().sum()))
prices.head()

## 1. Evolución histórica del precio y partición temporal

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(prices.index, prices["close"], color="black", lw=1)
for k, (a, b) in bounds.items():
    ax.axvspan(a, b, color=colors[k], alpha=0.12, label=k)
ax.set_title(f"{SYM} (cierre, escala log) y períodos train / val / test")
ax.legend(); plt.show()

## 2. Distribución de los retornos

In [ ]:
ret = np.log(prices["close"]).diff().dropna()
rows = {}
for k, (a, b) in bounds.items():
    r = ret.loc[a:b]
    rows[k] = dict(n=len(r), media_diaria=r.mean(), vol_anual=r.std() * np.sqrt(PPY),
                   asimetria=r.skew(), curtosis_exceso=r.kurt(), peor_dia=r.min(), mejor_dia=r.max())
print(pd.DataFrame(rows).T.round(4).to_string())

fig, axes = plt.subplots(1, 2)
r = ret.loc[bounds["train"][0]:bounds["train"][1]]
axes[0].hist(r, bins=120, density=True, alpha=0.7)
x = np.linspace(r.min(), r.max(), 400)
axes[0].plot(x, np.exp(-0.5 * ((x - r.mean()) / r.std()) ** 2) / (r.std() * np.sqrt(2 * np.pi)), "r", label="Normal")
axes[0].set_title("Retornos diarios (train) vs Normal: colas pesadas"); axes[0].legend()
h, edges = np.histogram(r, bins=120, density=True)
axes[1].semilogy((edges[:-1] + edges[1:]) / 2, h + 1e-6, ".")
axes[1].set_title("Mismo histograma, eje y logarítmico")
plt.tight_layout(); plt.show()

**Qué mirar:** curtosis en exceso alta y días extremos (−10 %, +10 %) → los retornos *no* son normales. Esto motiva
penalizar el drawdown y no solo el retorno medio: unos pocos días explican gran parte del riesgo.

## 3. Períodos de alta y baja volatilidad

In [ ]:
vol20 = ret.rolling(20).std() * np.sqrt(PPY)
thr = vol20.loc[bounds["train"][0]:bounds["train"][1]].quantile(0.8)
fig, ax = plt.subplots()
ax.plot(vol20.index, vol20, lw=0.8)
ax.axhline(thr, color="r", ls="--", label=f"percentil 80 de train = {thr:.0%}")
ax.set_title("Volatilidad realizada anualizada (ventana 20 días)"); ax.legend(); plt.show()

by_year = pd.DataFrame({"vol_anual": ret.groupby(ret.index.year).std() * np.sqrt(PPY),
                        "retorno_anual": ret.groupby(ret.index.year).sum()})
print(by_year.round(3).T.to_string())

## 4. Tendencias y caídas (drawdown del buy & hold)

In [ ]:
dd = drawdown_series(prices["close"])
fig, ax = plt.subplots()
ax.fill_between(dd.index, dd, color="tab:red", alpha=0.5)
for k, (a, b) in bounds.items():
    ax.axvspan(a, b, color=colors[k], alpha=0.08)
ax.set_title(f"Drawdown de {SYM} (comprar y mantener)"); plt.show()
for k, (a, b) in bounds.items():
    seg = prices["close"].loc[a:b]
    print(f"{k:>5}: retorno {seg.iloc[-1] / seg.iloc[0] - 1:+.1%} | máx. drawdown dentro del período {drawdown_series(seg).min():.1%}")

## 5. Volumen

In [ ]:
fig, ax = plt.subplots()
ax.plot(prices.index, prices["volume"] / 1e6, lw=0.5, alpha=0.6, label="volumen (M)")
ax.plot(prices.index, prices["volume"].rolling(20).mean() / 1e6, color="k", lw=1, label="media 20d")
ax.set_title(f"Volumen diario de {SYM} (en unidades del activo base)"); ax.legend(); plt.show()
# ¿sube el volumen cuando el mercado cae?
rv = pd.DataFrame({"abs_ret": ret.abs(), "vol_ratio": np.log(prices["volume"] / prices["volume"].rolling(20).mean())}).dropna()
print("Correlación |retorno| vs vol_ratio:", round(rv.corr().iloc[0, 1], 3))

## 6. Indicadores seleccionados

In [ ]:
raw = compute_features(prices)[cfg["data"]["features"]]
fig, axes = plt.subplots(len(raw.columns), 1, figsize=(11, 1.8 * len(raw.columns)), sharex=True)
for ax, c in zip(axes, raw.columns):
    ax.plot(raw.index, raw[c], lw=0.6); ax.set_ylabel(c, rotation=0, ha="right")
plt.tight_layout(); plt.show()

print("Estadísticos por período (features sin normalizar):")
print(pd.concat({k: raw.loc[a:b].describe().loc[["mean", "std"]] for k, (a, b) in bounds.items()}).round(4).to_string())

### Correlación entre features (redundancia) y relación con el retorno futuro

In [ ]:
train_raw = raw.loc[bounds["train"][0]:bounds["train"][1]]
corr = train_raw.corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns); ax.grid(False)
plt.colorbar(im); ax.set_title("Correlación de features (train)"); plt.show()

# SOLO ANÁLISIS: se usa el retorno de t+1 como etiqueta para medir poder predictivo.
# Esto NUNCA entra en el estado del agente.
fwd = np.log(prices["close"]).diff().shift(-1).reindex(train_raw.index)
ic = train_raw.apply(lambda s: s.corr(fwd, method="spearman")).sort_values()
print("Correlación de Spearman feature_t vs retorno_{t+1} (train):")
print(ic.round(4).to_string())

**Lectura esperada:** las correlaciones con el retorno del día siguiente son muy pequeñas (señal débil);
por eso es normal que el agente no supere claramente a buy & hold.
En cripto los drawdowns de buy & hold son enormes (−70 % o más), así que las features de volatilidad y tendencia pueden ayudar a **gestionar el riesgo**, que es donde un agente puede aportar valor.

## 7. Cambio de régimen entre train y test (distribution shift)

In [ ]:
z = pd.concat({k: s.features.describe().loc[["mean", "std"]] for k, s in splits.items()})
print("Features normalizadas con media/std de TRAIN. Si val/test se alejan de 0/1 hay cambio de régimen:")
print(z.round(2).to_string())

## 8. ¿Cuánto pesan los costos de transacción?

In [ ]:
cost = cfg["env"]["cost_bps"] / 1e4
print(f"Costo por cambiar de 0 % a 100 %: {cost:.2%}")
print(f"Retorno medio diario (train): {ret.loc[bounds['train'][0]:bounds['train'][1]].mean():.3%}")
print(f"Movimiento absoluto medio diario:     {ret.abs().mean():.3%}")
print(f"Un ida y vuelta (0→100→0) cuesta {2 * cost:.2%}, equivalente a ~{2 * cost / ret.loc[bounds['train'][0]:bounds['train'][1]].mean():.1f} días de retorno medio.")

## 9. Conclusiones (completar)

- **Estado:** …
- **Partición temporal:** …
- **Recompensa:** …
- **Riesgos de reward hacking anticipados:** …